In [1]:
# -*- coding: utf-8 -*-
"""
Corrected 2D DAE benchmark without RAR.

The mathematical model, network architecture, interface residual, periodic
condition, asymptotic reconstruction, LHS test set, timing, and metric
definitions are aligned with the corrected DAE-RAR implementation.

This no-RAR version preserves the original random-generation order:
    1. sample all 3000 fixed interior points;
    2. sample 1500 periodic-boundary time points;
    3. initialize the network.
No attempt is made to force the initial network weights to match the RAR run.
Only the total training loss is recorded, as in the original DAE benchmark.
"""

from __future__ import annotations

import math
import os
import random
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.spatial import cKDTree
from scipy.stats import qmc


# =============================================================================
# Basic settings
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01, 0.001, 0.0001]

X_MIN, X_MAX = -2.0, 2.0
Y_MIN, Y_MAX = -4.0, 4.0
T_FINAL = 1.0

METHOD_NAME = "DAE"
N_INTERNAL = 3000
N_B = 1500
EPOCHS = 15000
LEARNING_RATE = 1.0e-3

NUM_SAMPLES = 10000
LHS_SEED = 1234
EVAL_WARMUP = 20
EVAL_REPEAT = 200
BASE_PATH = "."
SAVE_LHS_PREDICTION = True

PI = torch.tensor(math.pi, dtype=DTYPE, device=DEVICE)


# =============================================================================
# Utilities
# =============================================================================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sync_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def mean_std(values) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    if len(arr) <= 1:
        return float(np.nanmean(arr)), 0.0
    return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=1))


# =============================================================================
# Network and asymptotic data
# =============================================================================
class Net(nn.Module):
    def __init__(self, layers: List[int]):
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.Linear(layers[i], layers[i + 1]) for i in range(len(layers) - 1)]
        )
        self.activation = nn.Tanh()
        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        for layer in self.layers[:-1]:
            z = self.activation(layer(z))
        return self.layers[-1](z)


def phi_minus_torch(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    expr = (
        16.0 * PI
        + 2.0 * PI * torch.cos(PI * (-x + y) / 4.0)
        + PI * x * torch.cos(PI * (-x + y) / 4.0)
        - 2.0 * torch.sin(PI * (-4.0 - x + y) / 4.0)
        + 2.0 * torch.sin(PI * (x + y) / 4.0)
    )
    return -torch.sqrt(torch.clamp(expr, min=1.0e-10)) / torch.sqrt(PI)


def phi_plus_torch(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    expr = (
        4.0 * PI
        - 2.0 * PI * torch.cos(PI * (-x + y) / 4.0)
        + PI * x * torch.cos(PI * (-x + y) / 4.0)
        - 2.0 * torch.sin(PI * (4.0 - x + y) / 4.0)
        + 2.0 * torch.sin(PI * (x + y) / 4.0)
    )
    return torch.sqrt(torch.clamp(expr, min=1.0e-10)) / torch.sqrt(PI)


def interface_residual(
    model: Net,
    y_t: torch.Tensor,
    create_graph: bool,
) -> torch.Tensor:
    """Residual of h_t - 0.5(h_y-1)(phi_minus+phi_plus)=0."""
    if y_t.ndim != 2 or y_t.shape[1] != 2:
        raise ValueError(f"y_t must have shape (N,2), got {tuple(y_t.shape)}")

    y = y_t[:, 0:1]
    t = y_t[:, 1:2]
    h = t * model(y_t)

    grads = torch.autograd.grad(
        h.sum(),
        y_t,
        create_graph=create_graph,
        retain_graph=create_graph,
        only_inputs=True,
    )[0]
    h_y = grads[:, 0:1]
    h_t = grads[:, 1:2]

    if create_graph:
        phi_m = phi_minus_torch(h, y)
        phi_p = phi_plus_torch(h, y)
        return h_t - 0.5 * (h_y - 1.0) * (phi_m + phi_p)

    # Kept identical to the RAR implementation for diagnostic/candidate use.
    h_v = h.detach()
    y_v = y.detach()
    return (
        h_t.detach()
        - 0.5
        * (h_y.detach() - 1.0)
        * (phi_minus_torch(h_v, y_v) + phi_plus_torch(h_v, y_v))
    ).detach()


def periodic_loss(model: Net, t_boundary: torch.Tensor) -> torch.Tensor:
    y_left = torch.full_like(t_boundary, Y_MIN)
    y_right = torch.full_like(t_boundary, Y_MAX)
    h_left = t_boundary * model(torch.cat([y_left, t_boundary], dim=1))
    h_right = t_boundary * model(torch.cat([y_right, t_boundary], dim=1))
    return torch.mean((h_left - h_right).square())


def total_training_loss(
    model: Net,
    y_t_internal: torch.Tensor,
    t_boundary: torch.Tensor,
) -> torch.Tensor:
    residual = interface_residual(model, y_t_internal, create_graph=True)
    return torch.mean(residual.square()) + periodic_loss(model, t_boundary)


def dae_reconstruct_gpu(
    model: Net,
    x_eval: torch.Tensor,
    y_eval: torch.Tensor,
    t_eval: torch.Tensor,
    mu: float,
) -> torch.Tensor:
    """Stable GPU reconstruction of the leading asymptotic solution."""
    if mu <= 0.0:
        raise ValueError(f"mu must be positive, got {mu}")

    with torch.enable_grad():
        y_in = y_eval.detach().clone().requires_grad_(True)
        t_in = t_eval.detach()
        h_val = t_in * model(torch.cat([y_in, t_in], dim=1))
        h_y = torch.autograd.grad(
            h_val.sum(), y_in, create_graph=False, retain_graph=False
        )[0]

    with torch.no_grad():
        h = h_val.detach()
        h_y_v = h_y.detach()

        phi_m_x = phi_minus_torch(x_eval, y_eval)
        phi_p_x = phi_plus_torch(x_eval, y_eval)
        phi_m_h = phi_minus_torch(h, y_eval)
        phi_p_h = phi_plus_torch(h, y_eval)

        delta = phi_p_h - phi_m_h
        b_factor = 1.0 - h_y_v

        eta_left = (h - x_eval) * delta * b_factor / (2.0 * float(mu))
        eta_right = (x_eval - h) * delta * b_factor / (2.0 * float(mu))

        # sigmoid(-z) = 1/(exp(z)+1), but avoids float32 overflow.
        u_left = phi_m_x + delta * torch.sigmoid(-eta_left)
        u_right = phi_p_x - delta * torch.sigmoid(-eta_right)
        return torch.where(x_eval <= h, u_left, u_right)


# =============================================================================
# Reference data and LHS test set
# =============================================================================
def get_target_col(df: pd.DataFrame) -> str:
    if "u" in df.columns:
        return "u"
    if "u0" in df.columns:
        return "u0"
    return str(df.columns[-1])


def true_solution_filename(mu: float) -> str:
    mu_id = int(round(-math.log10(mu)))
    return f"2d_U0_all_t_u_x_y_t_mu{mu_id}_101_101_101_Mathematica_620.csv"


def load_true_solution(mu: float) -> Tuple[pd.DataFrame, str]:
    filename = true_solution_filename(mu)
    path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find true solution file: {path}")

    df = pd.read_csv(path)
    df.columns = [str(col).lower().strip() for col in df.columns]
    required = {"t", "x", "y"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Reference file {filename} is missing columns: {sorted(missing)}")

    df = df.sort_values(by=["t", "x", "y"]).reset_index(drop=True)
    return df, filename


def generate_lhs_indices(df_true: pd.DataFrame, mu: float) -> np.ndarray:
    if len(df_true) < NUM_SAMPLES:
        raise ValueError("Reference grid has fewer rows than NUM_SAMPLES.")

    mins = [df_true["t"].min(), df_true["x"].min(), df_true["y"].min()]
    maxs = [df_true["t"].max(), df_true["x"].max(), df_true["y"].max()]
    tree = cKDTree(df_true[["t", "x", "y"]].to_numpy(dtype=np.float64))

    selected: List[int] = []
    used = set()
    for batch_id in range(100):
        if len(selected) >= NUM_SAMPLES:
            break
        sampler = qmc.LatinHypercube(d=3, seed=LHS_SEED + batch_id)
        scaled = qmc.scale(sampler.random(NUM_SAMPLES), mins, maxs)
        _, candidates = tree.query(scaled)
        for idx in candidates:
            idx_i = int(idx)
            if idx_i not in used:
                used.add(idx_i)
                selected.append(idx_i)
                if len(selected) == NUM_SAMPLES:
                    break

    if len(selected) < NUM_SAMPLES:
        remaining = np.setdiff1d(
            np.arange(len(df_true)), np.asarray(selected, dtype=np.int64)
        )
        fill = np.random.default_rng(LHS_SEED).choice(
            remaining, size=NUM_SAMPLES - len(selected), replace=False
        )
        selected.extend(int(v) for v in fill)

    indices = np.asarray(selected, dtype=np.int64)
    if len(indices) != NUM_SAMPLES or len(np.unique(indices)) != NUM_SAMPLES:
        raise RuntimeError("Failed to construct exactly NUM_SAMPLES unique LHS indices.")

    np.save(f"2d_LHS_sample_indices_mu{mu:.0e}.npy", indices)
    return indices


def build_or_load_lhs_test_set(mu: float) -> Dict[str, object]:
    df_true, filename = load_true_solution(mu)
    index_file = f"2d_LHS_sample_indices_mu{mu:.0e}.npy"
    sample_indices = None

    if os.path.exists(index_file):
        candidate = np.load(index_file)
        valid = (
            candidate.ndim == 1
            and len(candidate) == NUM_SAMPLES
            and len(np.unique(candidate)) == NUM_SAMPLES
            and int(candidate.min()) >= 0
            and int(candidate.max()) < len(df_true)
        )
        if valid:
            sample_indices = candidate.astype(np.int64, copy=False)
            print(f"[mu={mu}] Loaded valid LHS indices from {index_file}.")

    if sample_indices is None:
        sample_indices = generate_lhs_indices(df_true, mu)
        print(f"[mu={mu}] Generated {NUM_SAMPLES} LHS indices from {filename}.")

    chosen = df_true.iloc[sample_indices]
    t_np = chosen["t"].to_numpy(dtype=np.float64).reshape(-1, 1)
    x_np = chosen["x"].to_numpy(dtype=np.float64).reshape(-1, 1)
    y_np = chosen["y"].to_numpy(dtype=np.float64).reshape(-1, 1)
    true_np = chosen[get_target_col(df_true)].to_numpy(dtype=np.float64).reshape(-1)

    return {
        "t_lhs": torch.tensor(t_np, dtype=DTYPE, device=DEVICE),
        "x_lhs": torch.tensor(x_np, dtype=DTYPE, device=DEVICE),
        "y_lhs": torch.tensor(y_np, dtype=DTYPE, device=DEVICE),
        "true_lhs_np": true_np,
        "t_np": t_np,
        "x_np": x_np,
        "y_np": y_np,
        "n_test": len(sample_indices),
    }


def compute_error(true_u: np.ndarray, pred_u: np.ndarray) -> Tuple[float, float]:
    true_v = np.asarray(true_u, dtype=np.float64).reshape(-1)
    pred_v = np.asarray(pred_u, dtype=np.float64).reshape(-1)
    if true_v.shape != pred_v.shape:
        raise ValueError(
            f"true and predicted vectors have different shapes: "
            f"{true_v.shape} versus {pred_v.shape}"
        )

    diff = pred_v - true_v
    denom = np.linalg.norm(true_v)
    if denom == 0.0:
        raise ZeroDivisionError("Reference vector has zero Euclidean norm.")
    return float(np.linalg.norm(diff) / denom), float(np.max(np.abs(diff)))


# =============================================================================
# Main benchmark
# =============================================================================
def main() -> None:
    print("\n" + "=" * 88)
    print("Starting corrected 2D DAE benchmark without RAR")
    print(f"Device: {DEVICE} | N_test={NUM_SAMPLES} | LHS seed={LHS_SEED}")
    print(f"Fixed interior points: {N_INTERNAL}")
    print(f"Fixed optimization steps: {EPOCHS}")
    print("=" * 88 + "\n")

    lhs_data = {mu: build_or_load_lhs_test_set(mu) for mu in MU_LIST}
    metrics: Dict[float, List[Dict[str, object]]] = {mu: [] for mu in MU_LIST}

    for seed in SEEDS:
        print(f"\n--- Running seed={seed} ---")
        set_seed(seed)

        # Preserve the original no-RAR random-generation order.
        y_internal = Y_MIN + (Y_MAX - Y_MIN) * torch.rand(
            N_INTERNAL, 1, device=DEVICE, dtype=DTYPE
        )
        t_internal = T_FINAL * torch.rand(
            N_INTERNAL, 1, device=DEVICE, dtype=DTYPE
        )
        internal_batch = torch.cat([y_internal, t_internal], dim=1).detach()

        t_boundary = T_FINAL * torch.rand(
            N_B, 1, device=DEVICE, dtype=DTYPE
        )

        model = Net([2, 10, 10, 10, 10, 10, 1]).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        loss_history: List[torch.Tensor] = []
        total_point_steps = 0
        points_per_step = N_INTERNAL + 2 * N_B

        model.train()
        sync_cuda()
        train_start = time.perf_counter()

        for _ in range(EPOCHS):
            total_point_steps += points_per_step
            optimizer.zero_grad(set_to_none=True)

            # Isolate the input graph at every optimizer step.
            current_internal = (
                internal_batch.detach().clone().requires_grad_(True)
            )
            loss = total_training_loss(model, current_internal, t_boundary)
            loss.backward()
            optimizer.step()
            loss_history.append(loss.detach())

        sync_cuda()
        T_train = time.perf_counter() - train_start

        loss_hist = torch.stack(loss_history).cpu().numpy().astype(np.float64)
        e_loss = float(loss_hist[-1])
        np.save(f"2d_DAE_loss_history_seed{seed}.npy", loss_hist)

        T_train_per_iter_ms = 1.0e3 * T_train / EPOCHS
        T_train_per_iter_point_us = 1.0e6 * T_train / total_point_steps

        print(
            f" > trained: steps={EPOCHS}, fixed internal points={N_INTERNAL}, "
            f"T_train={T_train:.2f}s, e_loss={e_loss:.3e}"
        )

        model.eval()
        for mu in MU_LIST:
            data = lhs_data[mu]
            x_eval = data["x_lhs"]
            y_eval = data["y_lhs"]
            t_eval = data["t_lhs"]
            true_np = data["true_lhs_np"]

            for _ in range(EVAL_WARMUP):
                _ = dae_reconstruct_gpu(model, x_eval, y_eval, t_eval, mu)
            sync_cuda()

            eval_start = time.perf_counter()
            for _ in range(EVAL_REPEAT):
                _ = dae_reconstruct_gpu(model, x_eval, y_eval, t_eval, mu)
            sync_cuda()
            T_eval = (time.perf_counter() - eval_start) / EVAL_REPEAT

            prediction = dae_reconstruct_gpu(model, x_eval, y_eval, t_eval, mu)
            pred_np = prediction.detach().cpu().numpy().reshape(-1)
            e2, einf = compute_error(true_np, pred_np)
            T_total = T_train + T_eval

            print(
                f"    -> [mu={mu}] T_eval={T_eval:.6e}s, "
                f"e2={e2:.3e}, einf={einf:.3e}"
            )

            if SAVE_LHS_PREDICTION:
                pd.DataFrame(
                    {
                        "t": data["t_np"].reshape(-1),
                        "x": data["x_np"].reshape(-1),
                        "y": data["y_np"].reshape(-1),
                        "u": pred_np,
                    }
                ).to_csv(
                    f"2d_DAE_U0_predicted_LHS_mu{mu:.0e}_seed{seed}.csv",
                    index=False,
                )

            metrics[mu].append(
                {
                    "Seed": seed,
                    "N_test": int(data["n_test"]),
                    "e_loss": e_loss,
                    "e2": e2,
                    "einf": einf,
                    "T_train": T_train,
                    "T_eval": T_eval,
                    "T_total": T_total,
                    "T_train_per_iter_ms": T_train_per_iter_ms,
                    "T_train_per_iter_point_us": T_train_per_iter_point_us,
                    "total_trained_steps": EPOCHS,
                    "total_point_steps": total_point_steps,
                    "final_internal_points": N_INTERNAL,
                    "periodic_time_points": N_B,
                    "eval_warmup": EVAL_WARMUP,
                    "eval_repeat": EVAL_REPEAT,
                }
            )

    print("\n" + "=" * 88)
    print("ALL SEEDS COMPLETED. SUMMARY STATISTICS")
    print("=" * 88)

    for mu in MU_LIST:
        df = pd.DataFrame(metrics[mu])
        output = f"2d_DAE_mu{mu:.0e}_Metrics_Summary.csv"
        df.to_csv(output, index=False)

        cols = [
            "N_test",
            "e_loss",
            "e2",
            "einf",
            "T_train",
            "T_eval",
            "T_total",
            "T_train_per_iter_ms",
            "T_train_per_iter_point_us",
            "total_trained_steps",
            "total_point_steps",
            "final_internal_points",
        ]
        stats = {name: mean_std(df[name].values) for name in cols}

        print(f"\n### Results for corrected 2D DAE, mu={mu} ###")
        print(f"N_test: {stats['N_test'][0]:.0f} +/- {stats['N_test'][1]:.0f}")
        print(f"e_loss: {stats['e_loss'][0]:.3e} +/- {stats['e_loss'][1]:.3e}")
        print(f"e2: {stats['e2'][0]:.3e} +/- {stats['e2'][1]:.3e}")
        print(f"einf: {stats['einf'][0]:.3e} +/- {stats['einf'][1]:.3e}")
        print(f"T_train: {stats['T_train'][0]:.2f} +/- {stats['T_train'][1]:.2f} s")
        print(f"T_eval: {stats['T_eval'][0]:.6e} +/- {stats['T_eval'][1]:.6e} s")
        print(
            f"Optimization steps: {stats['total_trained_steps'][0]:.0f} +/- "
            f"{stats['total_trained_steps'][1]:.0f}"
        )
        print(
            f"Final internal points: {stats['final_internal_points'][0]:.0f} +/- "
            f"{stats['final_internal_points'][1]:.0f}"
        )
        print(f"Saved summary: {output}")


if __name__ == "__main__":
    main()



Starting corrected 2D DAE benchmark without RAR
Device: cuda | N_test=10000 | LHS seed=1234
Fixed interior points: 3000
Fixed optimization steps: 15000

[mu=0.01] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-02.npy.
[mu=0.001] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-03.npy.
[mu=0.0001] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-04.npy.

--- Running seed=33 ---
 > trained: steps=15000, fixed internal points=3000, T_train=232.39s, e_loss=8.558e-06
    -> [mu=0.01] T_eval=2.412985e-03s, e2=5.300e-03, einf=1.136e+00
    -> [mu=0.001] T_eval=2.400015e-03s, e2=1.651e-02, einf=3.909e+00
    -> [mu=0.0001] T_eval=2.399648e-03s, e2=2.546e-02, einf=5.995e+00

--- Running seed=99 ---
 > trained: steps=15000, fixed internal points=3000, T_train=232.61s, e_loss=1.628e-05
    -> [mu=0.01] T_eval=2.434908e-03s, e2=8.151e-03, einf=1.517e+00
    -> [mu=0.001] T_eval=2.451471e-03s, e2=1.484e-02, einf=4.346e+00
    -> [mu=0.0001] T_eval=2.412373e-03s, e2=2.627e